[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C56_Detection_Augmentation_Course/05_ablation/05_aug_ablation.ipynb)

# 05 · 增强的消融与验证（分场景切片 / 种子方差 / 配对检验 / 功效分析 / TSR 配方）

目标：把「这个增强有用吗」从一句直觉，变成一条**可以照着读数的流程**。

路线：**整体 mAP 的掩盖效应** → 种子方差与符号翻转率 → 配对设计与显著性 →
功效分析与 MDE → 增强过强的症状学 → **增强强度 × 训练时长的交互** →
**为 TSR 定制配方与验证方案** → ✏️ 4 道练习 → 📖 答案 → 🧪 工程胶囊。

> 心智模型：**平均会掩盖分布，单次会淹没在噪声，短训练会偏袒弱增强。**
> 这三件事各自都足以让你得出相反的结论 —— 而它们经常同时发生。

本 notebook 你会亲手实现：
1. 分场景切片对比：整体 +0.3 如何掩盖「晴天 −1.5、雨天 +3」，以及权重如何改变结论
2. 种子噪声下的**符号翻转率**与「挑最好种子」的作弊幅度
3. **配对 vs 非配对**的方差差异、配对 t 检验、bootstrap 置信区间
4. 功效分析（需要几个种子）与 **MDE**（这次实验能检出多小的提升）
5. 增强过强的症状学：容量-强度错配的定量演示
6. **增强强度 × 训练时长的交互**：同一比较在 12 与 300 epoch 下符号相反
7. **TSR 增强配方生成器** + 配方审计器 + 验证方案

## 1 · 整体 mAP 会掩盖什么

同一份实验数据，按里程加权和按安全风险加权，可以差 4 倍以上。
**「有没有用」这个问题在没有定义权重之前是不完整的。**

In [ ]:
import numpy as np, math, itertools, json

SCENES = ['晴天白天', '夜间', '雨天', '隧道']
MILEAGE = np.array([0.60, 0.20, 0.15, 0.05])          # 里程占比
RISK = np.array([1.0, 2.5, 2.0, 3.0])                 # 单位里程的安全风险倍数
AP_BASE = np.array([0.7200, 0.4800, 0.4100, 0.3900])
AP_NEW = np.array([0.7050, 0.5100, 0.4400, 0.4200])

delta = AP_NEW - AP_BASE
print('%-10s %8s %10s %10s %10s' % ('场景', '里程占比', 'baseline', '加增强后', '变化(pt)'))
for i, s in enumerate(SCENES):
    print('%-10s %7.0f%% %10.4f %10.4f %+10.2f'
          % (s, 100 * MILEAGE[i], AP_BASE[i], AP_NEW[i], 100 * delta[i]))

ov_base = float(MILEAGE @ AP_BASE)
ov_new = float(MILEAGE @ AP_NEW)
print('%-10s %8s %10.4f %10.4f %+10.2f'
      % ('里程加权', '100%', ov_base, ov_new, 100 * (ov_new - ov_base)))
assert abs(100 * (ov_new - ov_base) - 0.30) < 1e-9
assert abs(100 * delta[0] + 1.50) < 1e-9
print()
print('⚠️  只看最后一行 -> "+0.3，微弱正收益，上吧"。')
print('    真相是：占 60%% 里程的主力场景掉了 1.5 个点，被三个小场景的 +3 抬了回来。')

In [ ]:
# 换一套权重，同一份数据给出 4 倍不同的结论
w_mile = MILEAGE / MILEAGE.sum()
w_risk = (MILEAGE * RISK) / (MILEAGE * RISK).sum()

d_mile = float(w_mile @ delta) * 100
d_risk = float(w_risk @ delta) * 100
print('%-16s %s' % ('里程权重', np.round(w_mile, 4).tolist()))
print('%-16s %s' % ('风险权重', np.round(w_risk, 4).tolist()))
print()
print('里程加权的整体变化: %+.2f pt' % d_mile)
print('风险加权的整体变化: %+.2f pt   <- 是里程加权的 %.1f 倍' % (d_risk, d_risk / d_mile))
assert abs(d_mile - 0.30) < 1e-9
assert d_risk / d_mile > 4.0

# 回归门禁：任何切片跌破容差就拦截，与整体是否为正无关
def regression_gate(base, new, names, tol_pt=1.0):
    drops = [(n, 100 * (b2 - b1)) for n, b1, b2 in zip(names, base, new)
             if 100 * (b2 - b1) < -tol_pt]
    return {'blocked': len(drops) > 0, 'violations': drops}

g1 = regression_gate(AP_BASE, AP_NEW, SCENES, tol_pt=1.0)
g2 = regression_gate(AP_BASE, AP_NEW, SCENES, tol_pt=2.0)
print()
print('门禁 tol=1.0pt:', '❌ 拦截' if g1['blocked'] else '✅ 通过',
      [(n, round(v, 2)) for n, v in g1['violations']])
print('门禁 tol=2.0pt:', '❌ 拦截' if g2['blocked'] else '✅ 通过')
assert g1['blocked'] and not g2['blocked']
print()
print('✅ 规矩：**先定义切片与权重，再跑实验**。事后挑切法是无法自我察觉的 p-hacking。')

## 2 · 种子方差：多少提升才不是噪声

检测任务上，单次训练 mAP 的种子标准差典型在 **0.3–0.6 pt**。取 σ=0.5 算一笔账。

In [ ]:
SIGMA = 0.5          # 单次训练的种子标准差（pt）
rng = np.random.default_rng(0)
NT = 40000

def phi(z):          # 标准正态 CDF
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

sd_diff = math.sqrt(2) * SIGMA
print('单次训练 sigma = %.2f pt  ->  两次独立训练之差的 sd = %.3f pt' % (SIGMA, sd_diff))
print()

# (a) 零效应下观测到 |Δ| > 0.3 的概率
p_theory = 2 * (1 - phi(0.3 / sd_diff))
d_null = rng.normal(0, SIGMA, NT) - rng.normal(0, SIGMA, NT)
p_emp = float((np.abs(d_null) > 0.3).mean())
print('两个**完全相同**的配置各跑一次，|Δ| > 0.3 的概率:')
print('   理论 %.1f%%   实测 %.1f%%' % (100 * p_theory, 100 * p_emp))
assert abs(p_emp - p_theory) < 0.02 and abs(p_theory - 0.6714) < 0.002
print('   -> "+0.3 mAP" 这个数字，在单种子实验里几乎不含信息。')
print()

# (b) 真实效应 +0.4 时，单种子比较得出反号的概率
TRUE = 0.4
p_flip_theory = phi(-TRUE / sd_diff)
d_real = (TRUE + rng.normal(0, SIGMA, NT)) - rng.normal(0, SIGMA, NT)
p_flip = float((d_real < 0).mean())
print('真实效应 +%.1f pt，单种子比较给出**反号**的概率:' % TRUE)
print('   理论 %.1f%%   实测 %.1f%%' % (100 * p_flip_theory, 100 * p_flip))
assert abs(p_flip - p_flip_theory) < 0.02 and abs(p_flip_theory - 0.2858) < 0.002
print('   -> 约三分之一的概率，你会把一个真正有用的改动毙掉。')

In [ ]:
# 挑最好的种子：作弊幅度有多大
print('跑 k 个种子，只报最好的那个（哪怕真实效应为 0）:')
print('%-8s %14s %14s' % ('k', '平均"凭空收益"', '相对真实效应 +0.4'))
prev = -1e9
for k in (1, 2, 3, 5, 10, 20):
    best = rng.normal(0, SIGMA, size=(NT, k)).max(axis=1).mean()
    print('%-8d %+13.3f pt %13.2fx' % (k, best, best / 0.4))
    assert best > prev, '最大值的期望随 k 单调上升'
    prev = best
best10 = rng.normal(0, SIGMA, size=(NT, 10)).max(axis=1).mean()
assert 0.65 < best10 < 0.90
print()
print('⚠️  跑 10 个种子只报最好的，凭空得到 +%.2f pt —— **比绝大多数真实增强的效应还大**。' % best10)
print('    而且它不需要你有意作弊："这次跑崩了，重跑一遍" 就是一种选择性报告，')
print('    因为你只会对结果**差**的实验说"跑崩了"。')
print('✅ 防线：跑之前定死种子数与报告方式（均值±std），N 由功效分析决定，不由结果决定。')

## 3 · 配对设计：让方差自己抵消掉

`y[s,a] = mu[a] + b[s] + eps[s,a]`。同一个种子下 `b[s]` 对两个 arm 是共同的，做差就抵消。
**这是本模块性价比最高的一条建议。**

In [ ]:
SD_SEED, SD_RESID = 0.45, 0.15          # 种子效应 / 残差
MU = 0.40                                # 真实效应 (pt)
NS = 60000
r2 = np.random.default_rng(7)

b_shared = r2.normal(0, SD_SEED, NS)     # 配对：两个 arm 共用同一个种子效应
d_paired = (MU + b_shared + r2.normal(0, SD_RESID, NS)) \
           - (0.0 + b_shared + r2.normal(0, SD_RESID, NS))
d_unpaired = (MU + r2.normal(0, SD_SEED, NS) + r2.normal(0, SD_RESID, NS)) \
             - (0.0 + r2.normal(0, SD_SEED, NS) + r2.normal(0, SD_RESID, NS))

sd_p, sd_u = float(d_paired.std(ddof=1)), float(d_unpaired.std(ddof=1))
th_p = math.sqrt(2 * SD_RESID ** 2)
th_u = math.sqrt(2 * (SD_SEED ** 2 + SD_RESID ** 2))
print('%-24s %10s %10s' % ('设计', '实测 sd', '理论 sd'))
print('%-24s %10.4f %10.4f' % ('配对（共用种子）', sd_p, th_p))
print('%-24s %10.4f %10.4f' % ('非配对（各自种子）', sd_u, th_u))
assert abs(sd_p - th_p) < 0.01 and abs(sd_u - th_u) < 0.02
assert sd_u / sd_p > 2.5
print()
print('方差比 = %.1fx，标准差比 = %.1fx' % ((sd_u / sd_p) ** 2, sd_u / sd_p))
print('-> 同样的检出能力，配对设计所需的训练次数少了约 %.0f 倍。' % ((sd_u / sd_p) ** 2))
print()
print('⚠️  配对成立的两个前提（必须显式检查）：')
print('   ① 种子真的控制住了所有共享随机性（初始化 / shuffle / 增强 RNG）')
print('      —— 若 treatment 改变了增强算子数量，RNG 消耗次数就变了，后续序列错开，配对部分失效')
print('   ② 两个 arm 的其他一切完全相同（epoch / LR schedule / close-mosaic 时点 / 评测代码）')

In [ ]:
# 配对 t 检验 + bootstrap 置信区间：两个都要看
T_CRIT_95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365,
             8: 2.306, 9: 2.262, 10: 2.228, 11: 2.201, 12: 2.179, 13: 2.160,
             14: 2.145, 15: 2.131, 19: 2.093, 24: 2.064, 29: 2.045}

def paired_t(diffs):
    d = np.asarray(diffs, float); n = len(d)
    sd = float(d.std(ddof=1)); df = n - 1
    t = float(d.mean()) / (sd / math.sqrt(n)) if sd > 0 else float('inf')
    crit = T_CRIT_95.get(df, 1.960)
    return {'mean': float(d.mean()), 'sd': sd, 't': t, 'df': df,
            'crit': crit, 'significant': abs(t) > crit}

def bootstrap_ci(diffs, B=8000, alpha=0.05, seed=0):
    d = np.asarray(diffs, float); n = len(d)
    g = np.random.default_rng(seed)
    means = d[g.integers(0, n, size=(B, n))].mean(axis=1)
    return float(np.percentile(means, 100 * alpha / 2)), float(np.percentile(means, 100 * (1 - alpha / 2)))

# 两组差值，**均值都是 +0.40 pt**，唯一的区别是方差
D_PAIRED = [0.42, 0.31, 0.55, 0.28, 0.44]        # 配对设计得到的 5 个差值
D_UNPAIR = [0.95, -0.42, 1.10, -0.15, 0.52]      # 非配对设计得到的 5 个差值

print('%-14s %8s %8s %8s %8s %-10s %s' % ('设计', 'mean', 'sd', 't', 'crit', '显著?', '95% CI'))
for tag, d in [('配对', D_PAIRED), ('非配对', D_UNPAIR)]:
    r = paired_t(d); lo, hi = bootstrap_ci(d, seed=1)
    print('%-14s %+8.3f %8.3f %8.2f %8.3f %-10s [%+.3f, %+.3f]'
          % (tag, r['mean'], r['sd'], r['t'], r['crit'],
             '✅ 是' if r['significant'] else '❌ 否', lo, hi))

rp, ru = paired_t(D_PAIRED), paired_t(D_UNPAIR)
assert abs(rp['mean'] - 0.40) < 1e-9 and abs(ru['mean'] - 0.40) < 1e-9
assert rp['significant'] and not ru['significant']
lo_p, hi_p = bootstrap_ci(D_PAIRED, seed=1)
lo_u, hi_u = bootstrap_ci(D_UNPAIR, seed=1)
assert lo_p > 0, '配对：区间下界 > 0'
assert lo_u < 0 < hi_u, '非配对：区间跨过 0，证据不足'
assert (hi_u - lo_u) > 3 * (hi_p - lo_p)
print()
print('两组的均值**完全一样**（+0.40 pt），结论却相反 —— 差别只在方差。')
print('区间宽度：配对 %.3f  vs  非配对 %.3f （%.1f 倍）'
      % (hi_p - lo_p, hi_u - lo_u, (hi_u - lo_u) / (hi_p - lo_p)))
print('✅ 三步顺序：① 有没有效应(t) -> ② 效应多大多确定(CI) -> ③ 值不值得(工程容差)')

## 4 · 功效分析：动手之前先算需要多少种子

`n >= (z_{1-a/2} + z_{1-b})^2 * sd_d^2 / delta^2`，反过来是
`MDE(n) = (z+zb) * sd_d / sqrt(n)` —— **MDE 应该写在实验报告的最上面。**

In [ ]:
Z_ALPHA = {0.05: 1.959964, 0.10: 1.644854}
Z_POWER = {0.80: 0.841621, 0.90: 1.281552}

def seeds_needed(sigma_d, delta, alpha=0.05, power=0.80):
    k = (Z_ALPHA[alpha] + Z_POWER[power]) ** 2
    return int(math.ceil(k * sigma_d ** 2 / delta ** 2))

def mde(sigma_d, n, alpha=0.05, power=0.80):
    return (Z_ALPHA[alpha] + Z_POWER[power]) * sigma_d / math.sqrt(n)

SD_PAIRED, SD_UNPAIRED = 0.21, 0.67
print('alpha=0.05, power=0.80  ->  (z + z_beta)^2 = %.3f'
      % (Z_ALPHA[0.05] + Z_POWER[0.80]) ** 2)
print()
print('%-14s %8s %14s %14s %14s' % ('设计', 'sd_d', '检出+0.4', '检出+0.2', '检出+0.1'))
for tag, s in [('配对', SD_PAIRED), ('非配对', SD_UNPAIRED)]:
    print('%-14s %8.2f %13d个 %13d个 %13d个'
          % (tag, s, seeds_needed(s, 0.4), seeds_needed(s, 0.2), seeds_needed(s, 0.1)))
assert seeds_needed(0.21, 0.4) == 3
assert seeds_needed(0.21, 0.2) == 9
assert seeds_needed(0.21, 0.1) == 35
assert seeds_needed(0.67, 0.1) == 353
print()
print('%-24s %12s' % ('能跑的配对种子数', 'MDE (pt)'))
for n in (3, 5, 10, 20):
    print('%-24d %12.3f' % (n, mde(SD_PAIRED, n)))
assert abs(mde(0.21, 5) - 0.26311) < 1e-4
print()
print('-> 只能跑 5 个配对种子，MDE = %.2f pt。' % mde(SD_PAIRED, 5))
print('   **任何小于它的观测差值，无论正负，都不该被当成结论。**')
print()
print('要可靠检出 +0.1 pt：配对 %d 次训练，非配对 %d 次（12h/次 -> %.0f 天）。'
      % (seeds_needed(0.21, 0.1), seeds_needed(0.67, 0.1),
         seeds_needed(0.67, 0.1) * 0.5))
print('✅ 结论不是"想办法跑 353 次"，而是"+0.1 在当前方差下不可验证，不要基于它做决策"。')
print('   三条出路：① 降 sd_d（配对/确定性评测/更长训练）')
print('             ② 提 delta（做更有力的改动，别堆小 trick）')
print('             ③ 换信噪比更高的指标（分场景 AP 通常比整体 mAP 好）')

In [ ]:
# 多重比较：一次看 10 个算子 × 8 个切片会发生什么
def any_false_positive(n_tests, alpha=0.05):
    return 1 - (1 - alpha) ** n_tests

print('%-30s %14s' % ('检验次数', 'P(至少一个假阳性)'))
for n in (1, 5, 10, 40, 80):
    print('%-30d %13.1f%%' % (n, 100 * any_false_positive(n)))
assert abs(any_false_positive(10) - 0.4013) < 1e-3
assert any_false_positive(80) > 0.98
print()
print('10 个算子各做一次检验 -> 40%% 概率至少有一个假阳性。')
print('再乘 8 个切片 = 80 次检验 -> %.0f%% —— **纯噪声也会给你四个"显著"发现**。'
      % (100 * any_false_positive(80)))
print()

def bonferroni(pvals, alpha=0.05):
    m = len(pvals)
    return [p * m <= alpha for p in pvals]

pv = [0.001, 0.012, 0.030, 0.041, 0.049]
print('原始 p 值:', pv)
print('未校正显著:', [p <= 0.05 for p in pv], '-> %d 个' % sum(p <= 0.05 for p in pv))
print('Bonferroni:', bonferroni(pv), '-> %d 个' % sum(bonferroni(pv)))
assert sum(p <= 0.05 for p in pv) == 5 and sum(bonferroni(pv)) == 1
print()
print('✅ 规则：要么预先声明**一个**主指标（其余标探索性），要么做 Bonferroni/FDR 校正。')
print('   "我在 8 个切片里发现雨天显著提升" —— 若切片是事后挑的，几乎没有证据价值。')

## 5 · 增强过强的症状学

用一个可解析的玩具模型把「容量-强度错配」算出来：
增强提高有效任务难度（收敛更慢）、带来泛化收益（饱和）、但容量不足时会反噬。

In [ ]:
def run_model(capacity, aug, epochs, tau0=10.0):
    '''玩具模型：capacity=模型容量上限, aug=增强强度, epochs=训练预算。
       val_clean = 干净验证集上的表现； train_aug = 训练日志里看到的（训练集带增强）。'''
    tau = tau0 * (1 + 1.5 * aug)                     # 强增强 -> 收敛更慢
    prog = 1 - math.exp(-epochs / tau)               # 训练进度
    robust = 0.14 * (1 - math.exp(-2.0 * aug))       # 泛化收益（随强度饱和）
    penalty = 0.07 * aug ** 2 / capacity             # 容量不足时的反噬（∝ 强度^2 / 容量）
    val_clean = capacity * prog + robust - penalty
    return {'val_clean': val_clean, 'train_aug': val_clean - 0.22 * aug}

CAP_LARGE, CAP_TINY, EP = 0.78, 0.55, 100
AUGS = [0.0, 0.3, 0.6, 1.0, 1.4]

print('训练 %d epoch，两档模型容量下的表现' % EP)
print('%-6s | %-28s | %-28s' % ('aug', 'large (capacity 0.78)', 'tiny (capacity 0.55)'))
print('%-6s | %10s %10s %6s | %10s %10s %6s'
      % ('', 'train(带增强)', 'val(干净)', 'Δpt', 'train(带增强)', 'val(干净)', 'Δpt'))
base_L = run_model(CAP_LARGE, 0.0, EP)['val_clean']
base_T = run_model(CAP_TINY, 0.0, EP)['val_clean']
vals_L, vals_T = [], []
for a in AUGS:
    L, T = run_model(CAP_LARGE, a, EP), run_model(CAP_TINY, a, EP)
    vals_L.append(L['val_clean']); vals_T.append(T['val_clean'])
    print('%-6.1f | %10.4f %10.4f %+6.1f | %10.4f %10.4f %+6.1f'
          % (a, L['train_aug'], L['val_clean'], 100 * (L['val_clean'] - base_L),
             T['train_aug'], T['val_clean'], 100 * (T['val_clean'] - base_T)))

best_L = AUGS[int(np.argmax(vals_L))]
best_T = AUGS[int(np.argmax(vals_T))]
print()
print('最优强度: large = %.1f   tiny = %.1f' % (best_L, best_T))
assert best_L == 0.6 and best_T == 0.3
i1 = AUGS.index(1.0)
d_L = 100 * (vals_L[i1] - base_L); d_T = 100 * (vals_T[i1] - base_T)
print('同一份 aug=1.0 的配置: large %+.1f pt，tiny %+.1f pt —— **符号相反**' % (d_L, d_T))
assert d_L > 0 > d_T
print()
print('✅ 这就是 YOLO 系为 n/s/m/l/x 各档配不同增强强度的原因，不是调参玄学。')
print('   把大模型的增强配置照搬到 tiny 模型上，是很常见也很昂贵的错误。')

In [ ]:
# 症状诊断器：区分"正常现象"与"真的病了"
def diagnose(capacity, aug, epochs, tau0=10.0):
    base = run_model(capacity, 0.0, epochs, tau0)['val_clean']
    r = run_model(capacity, aug, epochs, tau0)
    return {
        'val_gt_train': r['val_clean'] > r['train_aug'],      # 症状：验证优于训练
        'val_below_baseline': r['val_clean'] < base,          # 判据：验证低于无增强 baseline
        'delta_pt': 100 * (r['val_clean'] - base),
    }

print('%-8s %-16s %-20s %10s %s' % ('aug', '验证>训练?', '验证<无增强baseline?', 'Δpt', '诊断'))
for a in AUGS:
    d = diagnose(CAP_LARGE, a, EP)
    verdict = ('✅ 健康' if not d['val_below_baseline'] else '❌ 增强过强')
    print('%-8.1f %-16s %-20s %+10.1f %s'
          % (a, '是（正常现象）' if d['val_gt_train'] else '否',
             '是' if d['val_below_baseline'] else '否', d['delta_pt'], verdict))

d06 = diagnose(CAP_LARGE, 0.6, EP)
d14 = diagnose(CAP_LARGE, 1.4, EP)
assert d06['val_gt_train'] and not d06['val_below_baseline'], 'aug=0.6：验证优于训练但完全健康'
assert d14['val_below_baseline'], 'aug=1.4：真的过强了'
print()
print('⚠️  「验证指标高于训练指标」在强增强的检测训练里是**预期行为**，不是 bug ——')
print('    训练时看到的是拼过/扭过/压暗加噪的图，验证时看到的是干净原图，后者当然更容易。')
print('    把它当成数据泄漏去查，会浪费大量时间。')
print('✅ 唯一的判据是 **验证指标 vs 无增强 baseline**，而不是「验证 vs 训练」。')

## 6 · 增强强度 × 训练时长：短训练下的比较是不公平的

**同一个比较，在 12 epoch 和 300 epoch 下符号相反。**
这是消融设计里最容易犯、后果最严重的错误。

In [ ]:
EPOCH_GRID = [12, 36, 100, 300]
AUG_GRID = [0.0, 0.3, 0.6, 1.0]
table = np.zeros((len(EPOCH_GRID), len(AUG_GRID)))
for i, ep in enumerate(EPOCH_GRID):
    for j, a in enumerate(AUG_GRID):
        table[i, j] = run_model(CAP_LARGE, a, ep)['val_clean']

print('%-12s' % '训练预算', end='')
for a in AUG_GRID:
    print('%12s' % ('aug=%.1f' % a), end='')
print('%14s' % '最优强度')
best_augs = []
for i, ep in enumerate(EPOCH_GRID):
    j = int(np.argmax(table[i]))
    best_augs.append(AUG_GRID[j])
    print('%-12s' % ('%d epoch' % ep), end='')
    for k in range(len(AUG_GRID)):
        mark = '*' if k == j else ' '
        print('%11.3f%s' % (table[i, k], mark), end='')
    print('%14.1f' % AUG_GRID[j])

print()
print('最优增强强度随预算变化:', best_augs)
assert best_augs == [0.0, 0.3, 0.6, 0.6]
assert all(best_augs[i] <= best_augs[i + 1] for i in range(len(best_augs) - 1)), '单调不减'

j0, j6 = AUG_GRID.index(0.0), AUG_GRID.index(0.6)
d12 = 100 * (table[0, j6] - table[0, j0])
d300 = 100 * (table[3, j6] - table[3, j0])
print()
print('同一个比较（aug=0.6 vs aug=0.0）:')
print('  12 epoch 预算下 : %+.2f pt  -> 结论"强增强有害"' % d12)
print('  300 epoch 预算下: %+.2f pt  -> 结论"强增强有益"' % d300)
assert d12 < -10.0 and d300 > 6.0
print()
print('⚠️  团队为了"快速迭代"把消融预算砍到 1/10，就会**系统性地**淘汰掉所有强增强方案 ——')
print('    短预算下的排序不是随机偏差，它有明确方向：**永远偏袒弱增强**。')
print('✅ 三条对策（成本递增）：')
print('   ① 用足够长的预算做消融（最可靠、最贵）')
print('   ② 缩短预算的同时按比例缩放增强强度（把"强度×时长"当联合超参）')
print('   ③ **看曲线不看终点**：强增强曲线在预算末尾仍明显上升 = "排序不可信"的证据')
print('      —— 第 ③ 条几乎不花额外成本，应当成为默认动作。')

In [ ]:
# 第 ③ 条的代码化：末段斜率检测器
def tail_slope(capacity, aug, epochs, window=0.2, tau0=10.0):
    '''最后 window 比例的训练区间里，val 还涨了多少（pt）。'''
    e0 = epochs * (1 - window)
    v0 = run_model(capacity, aug, e0, tau0)['val_clean']
    v1 = run_model(capacity, aug, epochs, tau0)['val_clean']
    return 100 * (v1 - v0)

print('%-10s %14s %14s %s' % ('aug', '12ep 末段斜率', '300ep 末段斜率', '12ep 是否已收敛'))
for a in AUG_GRID:
    s12, s300 = tail_slope(CAP_LARGE, a, 12), tail_slope(CAP_LARGE, a, 300)
    print('%-10.1f %+13.2f %+13.2f %s'
          % (a, s12, s300, '✅ 是' if s12 < 1.0 else '❌ 否，排序不可信'))
assert tail_slope(CAP_LARGE, 1.0, 12) > 1.0, '强增强在 12ep 时远未收敛'
assert tail_slope(CAP_LARGE, 1.0, 300) < 0.1, '300ep 时已平'
print()
print('✅ 12 epoch 下 aug>=0.3 的曲线全都还在明显上升 -> 这个预算下的排序不能采信。')
print('   把"末段斜率"打进实验报告，一眼就知道这次消融的结论有没有资格下。')

## 7 · 为 TSR 定制一套增强配方：按失效模式反推

**不要从「有哪些算子」出发去挑，要从「模型在哪里失效」出发去反推。**
优先级 = (目标 − 当前) × 里程占比 × 安全权重。

In [ ]:
FAILURE_MODES = [
    # id,             中文,                cur,  target, mileage, safety
    ('far_small',     '远距离小目标(<16px)', 0.31, 0.50, 0.35, 3.0),
    ('night',         '夜间 / 弱光',        0.48, 0.62, 0.18, 2.5),
    ('rain_fog',      '雨雾',              0.41, 0.58, 0.10, 2.0),
    ('rare_class',    '稀有类(施工/让行)',   0.36, 0.60, 0.04, 3.5),
    ('backlight',     '逆光 / 隧道出入口',   0.44, 0.60, 0.06, 2.5),
    ('motion_blur',   '运动模糊',           0.52, 0.65, 0.12, 1.5),
    ('billboard_fp',  '广告牌误检(precision)', 0.90, 0.96, 0.25, 1.2),
]

AUG_CATALOG = {
    'mosaic':        dict(targets=['far_small'], cpu_ms=26.0, p=1.0,
                          risk='中：必须配 close_mosaic'),
    'small_scale':   dict(targets=['far_small'], cpu_ms=0.4, p=0.5,
                          risk='低：主动制造小目标'),
    'motion_blur':   dict(targets=['motion_blur', 'far_small'], cpu_ms=1.2, p=0.2,
                          risk='中：过强会毁掉小目标的高频信息'),
    'lowlight':      dict(targets=['night', 'backlight'], cpu_ms=1.8, p=0.3,
                          risk='低：gamma + ISO 噪声 + 量化'),
    'hsv_mild':      dict(targets=['night', 'backlight'], cpu_ms=0.9, p=0.7,
                          risk='高：hue<=10，颜色是语义'),
    'atmos_fog':     dict(targets=['rain_fog'], cpu_ms=2.1, p=0.15,
                          risk='低：I = J*t + A(1-t)'),
    'copy_paste':    dict(targets=['rare_class'], cpu_ms=3.5, p=0.3,
                          risk='中：尺度/位置必须符合透视'),
    'hard_bg_paste': dict(targets=['billboard_fp'], cpu_ms=2.8, p=0.2,
                          risk='低：广告牌实例库贴成负样本'),
    'hflip':         dict(targets=[], cpu_ms=0.2, p=0.5,
                          risk='极高：左转->右转，必须类别白名单'),
}

def priority(m):
    _id, _cn, cur, tgt, mile, safe = m
    return (tgt - cur) * mile * safe

ranked = sorted(FAILURE_MODES, key=priority, reverse=True)
print('%-16s %-22s %8s %8s %8s %10s' % ('id', '失效模式', '缺口pt', '里程', '安全权重', '优先级'))
for m in ranked:
    print('%-16s %-22s %8.1f %7.0f%% %8.1f %10.4f'
          % (m[0], m[1], 100 * (m[3] - m[2]), 100 * m[4], m[5], priority(m)))
assert [m[0] for m in ranked[:3]] == ['far_small', 'night', 'rain_fog']
assert abs(priority(ranked[0]) - 0.1995) < 1e-9
print()
print('-> 排序结果和"哪个算子最流行"几乎没关系。**这正是它的价值。**')

In [ ]:
# 按优先级装配配方，并算算力账
def build_recipe(ranked_modes, catalog, decode_ms=8.0):
    chosen, cover = [], {}
    for m in ranked_modes:
        mid = m[0]
        for name, spec in catalog.items():
            if mid in spec['targets'] and name not in chosen:
                chosen.append(name)
                cover.setdefault(mid, []).append(name)
    cpu = decode_ms + sum(catalog[n]['cpu_ms'] for n in chosen)
    return {'ops': chosen, 'cover': cover, 'cpu_ms': cpu, 'decode_ms': decode_ms}

rec = build_recipe(ranked, AUG_CATALOG)
print('选中算子(%d 个):' % len(rec['ops']))
for n in rec['ops']:
    s = AUG_CATALOG[n]
    print('  %-16s p=%.2f  cpu=%5.1fms  targets=%-28s %s'
          % (n, s['p'], s['cpu_ms'], ','.join(s['targets']), s['risk']))
print()
print('单样本 CPU 成本: 解码 %.1f + 增强 %.1f = %.1f ms'
      % (rec['decode_ms'], rec['cpu_ms'] - rec['decode_ms'], rec['cpu_ms']))
assert set(rec['ops']) >= {'mosaic', 'small_scale', 'lowlight', 'atmos_fog', 'copy_paste'}
assert 'hflip' not in rec['ops'], 'hflip 不针对任何失效模式，不该被自动选入'
assert abs(rec['cpu_ms'] - 46.7) < 1e-9

# 算力账（复用模块 04 的公式）
GPU_IPS, CORES = 32 / 0.060, 16
def need_workers(cpu_ms):
    return int(math.ceil(GPU_IPS * cpu_ms / 1000.0))

print()
print('%-38s %10s %10s %8s' % ('方案', 'CPU(ms)', '最少worker', '16核可行?'))
plans = [('原样上（含 JPEG 解码）', rec['cpu_ms']),
         ('+ 预解码成未压缩格式', rec['cpu_ms'] - rec['decode_ms']),
         ('+ 预解码 + Mosaic 移到 GPU 侧', rec['cpu_ms'] - rec['decode_ms'] - 26.0)]
for tag, c in plans:
    w = need_workers(c)
    print('%-38s %10.1f %10d %8s' % (tag, c, w, '✅' if w <= CORES else '❌'))
assert need_workers(46.7) == 25 and need_workers(38.7) == 21 and need_workers(12.7) == 7
print()
print('⚠️  **不带算力账的增强配方是不可执行的** —— 这是训练平台上最常见的返工原因。')
print('    原样上要 25 个 worker，16 核机器给不出来；必须预解码 + Mosaic 上 GPU。')

In [ ]:
# 验证方案：每条失效模式配一个主指标切片 + 门禁 + 该切片的 MDE
def slice_sigma(sigma_overall, frac):
    '''切片越小，方差越大（样本量 ∝ frac）。'''
    return sigma_overall / math.sqrt(frac)

N_SEEDS = 5
SLICE_DEF = {'far_small': '像素尺寸桶 [<16,16-32,32-64,>64]',
             'night': '光照标签 night/dusk',
             'rain_fog': '天气标签 rain/fog',
             'rare_class': '类别族：尾部 20 类单列',
             'backlight': '场景标签 tunnel_in/tunnel_out',
             'motion_blur': '车速分桶',
             'billboard_fp': 'FP/km，按 FP 来源分类'}

print('配对 %d 种子，整体 sd_d = %.2f pt' % (N_SEEDS, SD_PAIRED))
print('%-14s %8s %9s %9s %10s %s' % ('失效模式', '切片占比', 'sd_slice',
                                     'MDE(5)', '缺口pt', '5 种子够吗'))
plan = []
for m in ranked:
    mid, _cn, cur, tgt, mile, _safe = m
    s = slice_sigma(SD_PAIRED, mile)
    md_ = mde(s, N_SEEDS)
    gap = 100 * (tgt - cur)
    ok = gap > md_
    plan.append((mid, s, md_, gap, ok))
    print('%-14s %7.0f%% %9.3f %9.3f %10.1f %s'
          % (mid, 100 * mile, s, md_, gap, '✅ 够' if ok else '❌ 不够，要加种子/扩切片'))

sig_rare = slice_sigma(SD_PAIRED, 0.04)
assert abs(sig_rare - 1.05) < 1e-9
assert abs(mde(sig_rare, 5) - 1.3158) < 1e-3
print()
print('稀有类切片只占 4%% 里程 -> sd 是整体的 %.1f 倍，MDE 高达 %.2f pt。'
      % (sig_rare / SD_PAIRED, mde(sig_rare, 5)))
print('要在这个切片上可靠检出 +0.5 pt，需要 %d 个配对种子 —— 不现实。'
      % seeds_needed(sig_rare, 0.5))
print('✅ 对策不是加种子，而是**扩大该切片的评测样本量**（定向采集/标注），')
print('   或者换一个方差更小的指标（尾部类召回 @ 固定 FP/km）。')
print()
print('验证方案（逐条上，不要一次全上）:')
for i, (mid, s, md_, gap, ok) in enumerate(plan[:4], 1):
    print('  %d. 加 %-28s -> 主指标 %s' % (i, ','.join(rec['cover'].get(mid, ['—'])),
                                          SLICE_DEF[mid]))
    print('     门禁：其余切片 95%%CI 下界 >= -0.5 pt；MDE=%.2f pt' % md_)
print('  5. 最后做一次全配方 vs baseline 的联合验证，检查"单条都涨、合起来不涨"')
print('     （那说明算子在争夺同一份模型容量，需要重新分配 p）')

## ✏️ 练习 1：分场景切片判定

实现 `slice_verdict(base, new, weights, names, tol_pt=1.0)`，返回
`{'overall_delta_pt', 'worst_slice', 'worst_delta_pt', 'blocked'}`。
`overall_delta_pt` 用 `weights` 加权（weights 会先归一化）；
`blocked` = 任一切片跌幅超过 `tol_pt`。

In [ ]:
def slice_verdict(base, new, weights, names, tol_pt=1.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
v = slice_verdict(AP_BASE, AP_NEW, MILEAGE, SCENES, tol_pt=1.0)
assert abs(v['overall_delta_pt'] - 0.30) < 1e-9, v
assert v['worst_slice'] == '晴天白天' and abs(v['worst_delta_pt'] + 1.50) < 1e-9
assert v['blocked'] is True
v2 = slice_verdict(AP_BASE, AP_NEW, MILEAGE, SCENES, tol_pt=2.0)
assert v2['blocked'] is False and abs(v2['overall_delta_pt'] - 0.30) < 1e-9
# 换成风险权重，整体收益是里程权重的 4 倍以上
v3 = slice_verdict(AP_BASE, AP_NEW, MILEAGE * RISK, SCENES, tol_pt=1.0)
assert v3['overall_delta_pt'] / v['overall_delta_pt'] > 4.0
# 全面上涨的情形
v4 = slice_verdict(AP_BASE, AP_BASE + 0.01, MILEAGE, SCENES)
assert not v4['blocked'] and abs(v4['overall_delta_pt'] - 1.0) < 1e-9
print('里程权重: 整体 %+.2f pt, 最差切片 %s %+.2f pt, 门禁 %s'
      % (v['overall_delta_pt'], v['worst_slice'], v['worst_delta_pt'],
         '拦截' if v['blocked'] else '通过'))
print('风险权重: 整体 %+.2f pt' % v3['overall_delta_pt'])
print('✅ 练习 1 通过：**只报整体 mAP 的实验报告应当被直接打回**')

## ✏️ 练习 2：配对检验的三步判定

实现 `paired_test(diffs, tol_pt)`，把三个问题分开回答，返回
`{'mean','ci','stat_sig','practically_sig','verdict'}`：
- `stat_sig`：配对 t 显著 **且** bootstrap 95% CI 不跨过 0
- `practically_sig`：CI 下界 > `tol_pt`（工程容差：值不值得做）
- `verdict`：`'证据不足：加种子'` / `'统计显著但不值得'` / `'采纳'`

（bootstrap 用 `bootstrap_ci(diffs, seed=1)`，保证可复现。）

In [ ]:
def paired_test(diffs, tol_pt):
    # TODO: 依次算 paired_t / bootstrap_ci / 工程容差，再给 verdict
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r1 = paired_test(D_PAIRED, tol_pt=0.20)
assert r1['stat_sig'] and r1['practically_sig'] and r1['verdict'] == '采纳', r1
r2 = paired_test(D_PAIRED, tol_pt=0.60)
assert r2['stat_sig'] and not r2['practically_sig']
assert r2['verdict'] == '统计显著但不值得', r2
r3 = paired_test(D_UNPAIR, tol_pt=0.20)
assert not r3['stat_sig'] and r3['verdict'] == '证据不足：加种子', r3
assert abs(r1['mean'] - 0.40) < 1e-9 and abs(r3['mean'] - 0.40) < 1e-9
print('%-34s %8s %10s %s' % ('输入', 'mean', 'CI下界', 'verdict'))
for tag, r in [('配对差值, 容差0.20', r1), ('配对差值, 容差0.60', r2),
               ('非配对差值, 容差0.20', r3)]:
    print('%-34s %+8.3f %+10.3f %s' % (tag, r['mean'], r['ci'][0], r['verdict']))
print('✅ 练习 2 通过：**统计显著 ≠ 值得做**，第三步才是真正的决策依据')

## ✏️ 练习 3：实验可行性规划

实现 `plan_experiment(sigma_d, delta_target, n_available, alpha=0.05, power=0.80)`，返回
`{'n_needed','mde','feasible','advice'}`。
`advice` 取 `'GO'`（`n_available >= n_needed`）或 `'NEED_MORE_SEEDS'`。
**跑之前算这一步，跑完就只剩照表读数。**

In [ ]:
def plan_experiment(sigma_d, delta_target, n_available, alpha=0.05, power=0.80):
    # TODO: 复用 seeds_needed 与 mde
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
a = plan_experiment(0.21, 0.4, 5)
assert a['n_needed'] == 3 and a['feasible'] and a['advice'] == 'GO'
assert abs(a['mde'] - 0.26311) < 1e-4
b = plan_experiment(0.21, 0.1, 5)
assert b['n_needed'] == 35 and not b['feasible'] and b['advice'] == 'NEED_MORE_SEEDS'
c = plan_experiment(0.67, 0.4, 5)
assert c['n_needed'] == 23 and not c['feasible']
assert abs(c['mde'] - 0.83945) < 1e-4
d = plan_experiment(1.05, 0.5, 5)          # 稀有类切片
assert d['n_needed'] == seeds_needed(1.05, 0.5) and not d['feasible']
print('%-40s %9s %9s %s' % ('场景', 'n_needed', 'MDE', 'advice'))
for tag, args in [('配对, 想检出+0.4, 有5种子', (0.21, 0.4, 5)),
                  ('配对, 想检出+0.1, 有5种子', (0.21, 0.1, 5)),
                  ('非配对, 想检出+0.4, 有5种子', (0.67, 0.4, 5)),
                  ('稀有类切片, 想检出+0.5, 有5种子', (1.05, 0.5, 5))]:
    r = plan_experiment(*args)
    print('%-40s %9d %9.3f %s' % (tag, r['n_needed'], r['mde'], r['advice']))
print('✅ 练习 3 通过：**MDE 应该写在实验报告的最上面**，它界定了这次实验能说什么')

## ✏️ 练习 4：TSR 增强配方审计器

实现 `audit_recipe(recipe)`，返回**排序后**的问题代码列表。规则：

| 代码 | 触发条件 |
|---|---|
| `FLIP_NO_WHITELIST` | `ops` 含 `hflip` 且 `p>0`，但 `flip_whitelist` 为空 |
| `HUE_TOO_STRONG` | 任一算子的 `hue_deg > 10`（颜色是语义） |
| `MOSAIC_NO_CLOSE` | `ops` 含 `mosaic` 但 `close_mosaic_epochs <= 0` |
| `AUG_TOO_SPARSE` | `prod(1-p) > 0.25`（太多样本完全没被增强） |
| `CPU_OVER_BUDGET` | `cpu_ms > cpu_budget_ms` |
| `MODE_UNCOVERED` | `top_modes` 里有不在 `covered_modes` 的 |
| `NO_VAL_SLICE` | `val_slices` 为空 |

In [ ]:
def audit_recipe(recipe):
    # TODO: 逐条检查，返回 sorted(codes)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
BAD = {'ops': {'hflip': {'p': 0.5}, 'hsv': {'p': 0.5, 'hue_deg': 25},
               'mosaic': {'p': 1.0}},
       'flip_whitelist': None, 'close_mosaic_epochs': 0,
       'cpu_ms': 52.0, 'cpu_budget_ms': 30.0,
       'top_modes': ['far_small', 'night', 'rain_fog'], 'covered_modes': ['far_small'],
       'val_slices': []}
GOOD = {'ops': {'hflip': {'p': 0.5}, 'hsv': {'p': 0.7, 'hue_deg': 8},
                'mosaic': {'p': 1.0}, 'lowlight': {'p': 0.3}, 'copy_paste': {'p': 0.3}},
        'flip_whitelist': ['circle_speed', 'warning_tri'], 'close_mosaic_epochs': 15,
        'cpu_ms': 12.7, 'cpu_budget_ms': 30.0,
        'top_modes': ['far_small', 'night', 'rain_fog'],
        'covered_modes': ['far_small', 'night', 'rain_fog', 'rare_class'],
        'val_slices': ['size_bucket', 'light', 'weather']}
SPARSE = dict(GOOD, ops={'lowlight': {'p': 0.1}, 'atmos_fog': {'p': 0.1},
                         'motion_blur': {'p': 0.1}})

assert audit_recipe(BAD) == ['CPU_OVER_BUDGET', 'FLIP_NO_WHITELIST', 'HUE_TOO_STRONG',
                             'MODE_UNCOVERED', 'MOSAIC_NO_CLOSE', 'NO_VAL_SLICE'], audit_recipe(BAD)
assert audit_recipe(GOOD) == [], audit_recipe(GOOD)
assert audit_recipe(SPARSE) == ['AUG_TOO_SPARSE'], audit_recipe(SPARSE)
print('BAD   ->', audit_recipe(BAD))
print('GOOD  ->', audit_recipe(GOOD) or '无问题 ✅')
print('SPARSE->', audit_recipe(SPARSE))
print('✅ 练习 4 通过：约束注释必须进 CI，不能指望下一个人记得')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def slice_verdict(base, new, weights, names, tol_pt=1.0):
    base = np.asarray(base, float); new = np.asarray(new, float)
    w = np.asarray(weights, float); w = w / w.sum()
    d_pt = 100 * (new - base)
    i = int(np.argmin(d_pt))
    return {'overall_delta_pt': float(w @ d_pt),
            'worst_slice': names[i],
            'worst_delta_pt': float(d_pt[i]),
            'blocked': bool(d_pt.min() < -tol_pt)}

In [ ]:
# 练习 2 参考答案
def paired_test(diffs, tol_pt):
    r = paired_t(diffs)
    lo, hi = bootstrap_ci(diffs, seed=1)
    stat = bool(r['significant'] and (lo > 0 or hi < 0))
    prac = bool(lo > tol_pt)
    if not stat:
        verdict = '证据不足：加种子'
    elif not prac:
        verdict = '统计显著但不值得'
    else:
        verdict = '采纳'
    return {'mean': r['mean'], 'ci': (lo, hi), 'stat_sig': stat,
            'practically_sig': prac, 'verdict': verdict}

In [ ]:
# 练习 3 参考答案
def plan_experiment(sigma_d, delta_target, n_available, alpha=0.05, power=0.80):
    n_need = seeds_needed(sigma_d, delta_target, alpha, power)
    m = mde(sigma_d, n_available, alpha, power)
    ok = n_available >= n_need
    return {'n_needed': n_need, 'mde': m, 'feasible': ok,
            'advice': 'GO' if ok else 'NEED_MORE_SEEDS'}

In [ ]:
# 练习 4 参考答案
def audit_recipe(recipe):
    ops = recipe.get('ops', {})
    codes = []
    if ops.get('hflip', {}).get('p', 0) > 0 and not recipe.get('flip_whitelist'):
        codes.append('FLIP_NO_WHITELIST')
    if any(spec.get('hue_deg', 0) > 10 for spec in ops.values()):
        codes.append('HUE_TOO_STRONG')
    if 'mosaic' in ops and recipe.get('close_mosaic_epochs', 0) <= 0:
        codes.append('MOSAIC_NO_CLOSE')
    p_none = 1.0
    for spec in ops.values():
        p_none *= (1.0 - spec.get('p', 0.0))
    if p_none > 0.25:
        codes.append('AUG_TOO_SPARSE')
    if recipe.get('cpu_ms', 0) > recipe.get('cpu_budget_ms', float('inf')):
        codes.append('CPU_OVER_BUDGET')
    if set(recipe.get('top_modes', [])) - set(recipe.get('covered_modes', [])):
        codes.append('MODE_UNCOVERED')
    if not recipe.get('val_slices'):
        codes.append('NO_VAL_SLICE')
    return sorted(codes)

---
## 🧪 真实工程胶囊：实验计划模板 + 分场景报告 + TSR 配方

In [ ]:
RECIPE = r'''
# ============ 1. 实验计划（**跑之前写死并入库**）============
plan:
  hypothesis : 低光合成能提升夜间 AP，且不损害晴天 AP
  primary    : 夜间切片 AP                       # ★ 只有一个主指标
  gate       : 晴天切片 AP 的 95%CI 下界 >= -0.5 pt   # ★ 非劣性，用区间下界不用 p 值
  design     : paired, seeds=[0,1,2,3,4]          # ★ 配对：方差降 10 倍
  identical  : epochs / lr_schedule / close_mosaic_epoch / eval_code 全同
  mde        : (1.96+0.84) * sd_d / sqrt(5)       # ★ 写在报告最上面
  slices     : weather x4 / size_bucket x4 / class_family x5   (slices_v3, 已入库)
  n_locked   : true                                # ★ 不许中途加种子

# ============ 2. 跑之前的三条前置断言 ============
assert eval(model, val) == eval(model, val)        # 评测必须纯确定性（模块 04）
assert aug_fingerprint(cfg_A) != aug_fingerprint(cfg_B)   # 确实改了增强
assert diff_only_touches(cfg_A, cfg_B, allow={'aug'})     # ★ 单变量

# ============ 3. 分场景报告（**只报整体 mAP 的报告应被打回**）============
for slice_name in SLICES:
    d = [ap_new[s][slice_name] - ap_base[s][slice_name] for s in SEEDS]   # 逐种子配对差
    t   = paired_t(d)                        # 有没有效应
    ci  = bootstrap_ci(d)                    # 效应多大、多确定
    print(slice_name, mean=t['mean'], ci=ci, t=t['t'], crit=t['crit'],
          vs_mde=t['mean']/MDE)              # ★ 效应量 vs MDE 比 p 值有用
# 多重比较：主指标之外全部标 exploratory，或做 Bonferroni/FDR

# ============ 4. TSR 增强配方 v1（按失效模式反推，★=约束来源）============
stage_A_mixing:
  mosaic:       p=1.0  scale=(0.4,1.2)  close_at_epoch=-15   # ★m03 close-mosaic
  mixup:        p=0.08                                        # 小模型档置 0（容量-强度错配）
stage_B_geometric:                        # ★m04 合成一个仿射矩阵，只重采样一次
  affine:       p=0.9  scale=(0.5,1.5) translate=0.1 rotate=(-8,8) shear=(-4,4)
  small_scale:  p=0.5  extra_downscale=(0.5,0.8)   # ← 主动制造小目标（far_small）
  hflip:        p=0.5  whitelist=SYMMETRIC_CLASSES # ★m01 左转/右转/文字牌禁翻
stage_C_cleanup:
  clip + min_area=16px^2 + min_visibility=0.3      # ★m04 紧跟几何段
stage_D_photometric:                      # ★m04 必须在几何之后（否则被插值抹平）
  lowlight:     p=0.30 gamma=(1.6,2.6) iso_noise=(8,30) quant_bits=7   # night
  atmos_fog:    p=0.15 beta=(0.4,1.2) A=(0.7,0.95)                     # rain_fog
  hsv:          p=0.70 hue=+-8 sat=+-25 val=+-25   # ★m02 hue<=10：颜色是语义
  motion_blur:  p=0.20 length=(5,15) angle~车速与转向
stage_E_instance:
  copy_paste_rare:   p=0.30 classes=TAIL_20 scale_by_perspective=True  # rare_class
  copy_paste_hardbg: p=0.20 source=广告牌实例库 as_negative=True        # billboard_fp
stage_F_normalize:
  letterbox(pad=114, stride=32) -> RGB -> /255 -> mean/std   # ★与 val/部署共用同一函数

# ============ 5. 算力账（不带算力账的配方不可执行）============
# 增强 38.7ms + 解码 8.0ms = 46.7ms/样本 -> 需要 25 个 worker（16 核给不出）
# 对策：预解码成未压缩格式（-8ms）+ Mosaic 移到 GPU 侧（-26ms）-> 12.7ms -> 7 个 worker ✅

# ============ 6. 验证顺序（逐条上，不要一次全上）============
# 1) small_scale+mosaic -> 主指标 size_bucket<16px      2) lowlight+hsv -> 光照切片
# 3) atmos_fog -> 天气切片                              4) copy_paste -> 尾部类召回@固定FP/km
# 5) 全配方联合验证：检查"单条都涨、合起来不涨"（算子在争夺同一份容量）
# 门禁：其余切片 95%CI 下界 >= -0.5 pt；专项门禁：跨颜色族错分率（mAP 对它不敏感）
# 报告代价：t_aug / min_workers / 训练时长 / close-mosaic 后曲线是否已平（末段斜率）
'''
print(RECIPE)
for k in ['primary', 'gate', 'paired', 'mde', 'n_locked', 'aug_fingerprint',
          'close_at_epoch', 'whitelist', 'hue<=10', 'min_visibility',
          'copy_paste_rare', '25 个 worker', '末段斜率']:
    assert k in RECIPE, k
print('✅ 配方覆盖：实验计划 · 前置断言 · 分场景报告 · TSR 配方 · 算力账 · 验证顺序')

### 小结

- **整体 mAP 是加权平均，平均的天职就是抹掉分布**。+0.3 可能是「晴天 −1.5、雨天 +3」；
  换一套权重（里程 vs 安全风险）同一份数据能差 **4 倍**。
  **先定义切片与权重，再跑实验**——事后挑切法是无法自我察觉的 p-hacking。
- **种子方差**：σ=0.5 pt 时，两个完全相同的配置有 **67%** 概率给出 |Δ|>0.3；
  真实效应 +0.4 时单种子有 **29%** 概率给出反号；跑 10 个种子只报最好的，
  凭空得到 **+0.77 pt**——比大多数真实增强的效应还大。
- **配对设计**是性价比最高的一招：共用同一批种子，差值 sd 从 0.67 降到 0.21，
  **方差降 10 倍、算力降 8 倍**。两组均值同为 +0.40 pt，配对显著、非配对不显著。
- **三步顺序**：有没有效应（t）→ 效应多大多确定（bootstrap CI）→ 值不值得（工程容差）。
  **MDE 写在报告最上面**；「不劣化」类要求必须用**区间下界**判，不能用 p 值。
- **增强过强的判据是「验证 vs 无增强 baseline」**，不是「验证 vs 训练」——
  后者在强增强下是**预期行为**。容量-强度错配：同一份 aug=1.0 在 large 上 +1.7、tiny 上 −1.6。
- **短训练系统性偏袒弱增强**：aug=0.6 vs 0.0 在 12 epoch 下 −11.4 pt、300 epoch 下 +6.6 pt。
  低成本对策：**看末段斜率**——曲线还在涨就说明这次消融没资格下结论。
- **TSR 配方按失效模式反推**：优先级 =(缺口 × 里程 × 安全权重)，
  远距离小目标 ≫ 夜间 > 雨雾 ≈ 稀有类。每条算子都要带**约束来源**与**验证切片**，
  并且**必须带算力账**（46.7 ms/样本 = 25 个 worker，16 核跑不动）。

至此 C56 全部结束。下一站建议：**C57（小目标检测）**——本模块排在第一优先级的
「远距离小目标」缺口，在那里有系统的架构与损失层面的解法。